# False-Positive-Constrained IoT Intrusion Detection

This notebook reviews the retained outputs of a reproducible pilot study on the CICIoT2023 dataset. The raw data are not redistributed. The full pipeline can be rerun with the scripts in this repository after placing the verified dataset shard at `data/ciciot2023/Merged01.csv`.

## Data source and input identity

Official source: https://www.unb.ca/cic/datasets/iotdataset-2023.html  
Article: https://doi.org/10.3390/s23135941  
Input SHA-256: `8b43d6552a8cafd3b0ca2cedf6464ca3fe644d7fc9bb5dfe906368f1542792fe`

## Dataset audit

In [1]:
import json, pandas as pd
audit = json.load(open('results/g1_audit.json'))
# Compact audit table assembled from the retained JSON record.
audit_table

                      Item  Observed
                      Rows    712311
                Predictors        39
      Exact duplicate rows    167368
    Duplicate feature rows    194204
Conflicting feature groups        16
             Missing cells        22
            Infinite cells        14


The audit motivated feature-level deduplication before splitting. Groups with conflicting binary labels were excluded from the primary analysis.

## Baselines at the default threshold

In [2]:
baseline = pd.read_csv('results/g2_results_table.csv')
baseline.loc[baseline.partition.eq('test'), ['model','recall_attack','false_positive_rate','mcc','macro_average_precision']]

              model  recall_attack  false_positive_rate      mcc  macro_average_precision
logistic_regression       0.968796             0.000302 0.705478                 0.884645
      random_forest       0.983910             0.018429 0.802775                 0.946267


## Validation-constrained selection

In [3]:
selected = pd.read_csv('results/g3_selected_results_table.csv')
selected.loc[selected.selected, ['model','partition','threshold','recall_attack','false_positive_rate','mcc']]

        model  partition  threshold  recall_attack  false_positive_rate      mcc
random_forest validation    0.59853       0.982285             0.009668 0.793974
random_forest       test    0.59853       0.981667             0.012085 0.787518


![Validation selection and computational cost](figures/g3_operating_point_tradeoff.png)

## Conditional uncertainty intervals

In [4]:
intervals = pd.read_csv('results/g4_confidence_intervals.csv')
intervals.loc[(intervals.partition == 'test') & intervals.metric.isin(['false_positive_rate','recall_attack','balanced_accuracy','mcc'])]

             metric  observed  ci_95_lower  ci_95_upper
      recall_attack  0.981667     0.980839     0.982484
false_positive_rate  0.012085     0.008459     0.016012
  balanced_accuracy  0.984791     0.982851     0.986620
                mcc  0.787518     0.780557     0.794510


![Conditional uncertainty intervals](figures/g4_uncertainty_stability.png)

## Interpretation

The selected Random Forest reached 98.167% attack recall and 1.208% FPR on the held-out partition. Its validation FPR was below 1%, but the held-out point estimate was not. The intervals condition on the fitted model, selected threshold, and observed partitions; they do not include training or model-selection variability.

A stronger follow-up should use temporal or device-group separation, an external dataset, repeated training runs, and confidence-aware threshold selection.

## Full rerun

```bash
python scripts/g1_audit.py
python scripts/g2_baselines.py
python scripts/g3_pareto_selection.py
python scripts/g4_uncertainty.py
```